# Análisis de usuarios con SQL: Coding together

## Objetivo

---

El objetivo principal de este ejercicio es analizar datos de comportamiento de usuarios durante su interacción con una plataforma digital (web o mobile). Los datos incluyen información básica de usuario y un registro cronológico de eventos que representan las acciones que realiza dentro del sitio. 

Los objetivos particulares seran:
- Evaluar el comportamiento de adquisición, uso y retorno de usuarios.
- Analizar fricción dentro del journey (puntos donde los usuarios típicamente abandonan).
- Estimar tasas de retención entre sesiones.
- Generar insights para optimizar la experiencia del usuario o estrategias de marketing.
- Usar consultas SQL para extraer los datos necesarios

## 🏢 Contexto del problema

---

Una empresa de **e-commerce de productos para el hogar** llamada *HomeZone* quiere comprender cómo los usuarios interactúan con su plataforma digital. La empresa ha observado que, aunque muchos usuarios visitan el sitio y navegan productos, no todos llegan a completar una compra. 

Además, la dirección sospecha que:
- Los usuarios **móviles** pueden estar teniendo una experiencia de compra más lenta.
- Existen **zonas geográficas** donde la retención es más baja.
- El proceso de pago podría estar introduciendo **fricción**.

Para analizar esto, tenemos acceso a la base de datos `User Journey` donde las tablas `users` y `events` contienen la información necesaria


## 🎲 Dinámica 

---

Nos dividiremos en equipos para efectuar el análisis. Cada equipo tiene un formato donde irá registrando el proceso completo del análisis [Reporte User Journey](https://docs.google.com/spreadsheets/d/17mZhu4FvXXW6tQCU2TncvkbN3K6bk4HURIsw7DnA8uU/copy)


## 🔍 Análisis exploratorio

---

Antes de evaluar embudos o cohortes, es importante conocer **la estructura de la base de usuarios**.

### Calidad de datos

1. Consistencia de datos
   - Duplicados
   - Nulos
   - Valores por columna

### Datos a explorar:

1. **Globales**
   -Total de registros en las tablas `events` y `users`
   - Ventana de fechas 
   - Usuarios únicos

1. **Demografía**

   - Proporción por sexo (F/M).
   - Distribución por zona: `North`, `South`, `West`, `East`.
   - Preferencia de acceso: `web` vs `mobile` 📱🖥️
   - *(Opcional)* Distribución por grupos de edad (`menos de 25, menos de 45, menos de 60, más de 60`).

   - **Consulta de referencía**

   ```sql
   -- Cantidad de usuarios por sexo
   SELECT sex, COUNT(DISTINCT id) as n_usuarios
   FROM users
   GROUP BY sex
   ORDER BY n_usuarios DESC
   ```

3. **Eventos**
   - Conteo total y usuarios únicos por tipo de evento.


2. **Actividad**
   - (*Opcional*) Tiempo promedio entre retornos a `sesion_start` (indicador de retención).



## 🧭 Análisis de embudo (User Journey → Conversión)

---

El proceso típico hacia la conversión puede representarse como:

| Etapa | Evento | Indicador clave |
|------|---------|----------------|
| 1 | `sesion_start` | Usuarios que ingresan |
| 2 | `search_page` | Usuarios con intención de exploración |
| 3 | `product_page` | Evaluación de productos |
| 4 | `car_checkout` | Preparación de compra |
| 5 | `payment_confirmation` | Conversión completada |



### 📉 Cálculo de tasas

**Tasa de paso entre etapas**
$$
\text{Tasa\_paso}_{i \rightarrow j} = \frac{\text{Usuarios que llegan a j}}{\text{Usuarios que llegan a i}}
$$

**Conversión total**
$$
\text{Conversión} = \frac{\text{Usuarios que completan pago}}{\text{Usuarios que inician sesión}}
$$

### 🔥 Posibles insights
- Si muchos usuarios abandonan en `product_page → car_checkout`, puede indicar **dudas de valor o precio**.
- Si el abandono ocurre en `car_checkout → payment_confirmation`, probablemente hay **fricción en el pago** (UX, métodos limitados, errores).


**Completa la consulta para el embudo de conversion**


```sql 
-- Resumen para todos las etapa

SELECT event_name, 
    COUNT (*) as n_events,
    - - -  as user_unique
    FROM events
    - - - 

```

```sql 
-- Resumen de usuarios navegando por etapas

WITH start_session AS (
    SELECT  DISTINCT user_id FROM events WHERE event_name='sesion_start'
),
search_page AS (
    SELECT DISTINCT user_id FROM events WHERE event_name='sesion_start' and user_id in (SELECT user_id FROM start_session)
),
product_page AS()
car_checkout AS()
payment_confirmation AS()

SELECT  
(SELECT COUNT(*) FROM search_page) as user_search_page
 ()   as user_search_page
 ()   as user_product_page
 ()   as user_car_checkout
 ()   as user_payment_confirmation

```







## 🔁 Análisis de Cohorte (Retención)


---



Las cohortes se definen por la **fecha de primera visita (first_visit)** agrupada semanal o mensualmente.

| Cohorte (Alta) | # Usuarios | Retención Día 1 | Retención Semana 1 | Conversión |
|----------------|-----------|----------------|-------------------|------------|
| 2024-01 | 450 | 32% | 14% | 6% |
| 2024-02 | 510 | 29% | 11% | 7% |

### 📌 Métrica principal de retención

Se considera que un usuario de la cohorte  tiene actividad despues de su registro.

**Retención por cohorte**
$$
\text{Retención}(t) = \frac{\text{Usuarios activos en } t}{\text{Usuarios en cohorte inicial}}
$$

### 🎯 Preguntas clave
- ¿Qué cohortes semanales muestran mejor retencion?
- ¿Que se observa de la retención observando la *región* y el tipo de *platform*



**Query de referencia**


```sql
WITH base AS (
    SELECT
        id,
        first_visit,
        CAST(strftime('%d', first_visit) AS INT) AS day
    FROM users
),

-- Cohortes semanales
user_cohort AS (
    SELECT
        id,
        CASE
            WHEN day <= 7 THEN strftime('%Y-%m-', first_visit) || '01'
            WHEN day <= 14 THEN strftime('%Y-%m-', first_visit) || '08'
             WHEN day <= 21 THEN strftime('%Y-%m-', first_visit) || '15'
            ELSE strftime('%Y-%m-', first_visit) || '22'
        END AS cohort_date
    FROM base
),

-- Eventos asignados a cohortes
cohort_events AS (
    SELECT
        uc.id AS user_id,
        uc.cohort_date,
        ee.event_date,
        CAST((julianday(ee.event_date) - julianday(uc.cohort_date)) / 7 AS INT) AS offset_week
    FROM user_cohort uc
    LEFT JOIN events ee
        ON uc.id = ee.user_id
),

-- Contamos usuarios únicos por cohorte y offset
cohort_counts AS (
    SELECT
        cohort_date,
        offset_week,
        COUNT(DISTINCT user_id) AS users_cohort_offset
    FROM cohort_events
    GROUP BY cohort_date, offset_week
),

-- Contamos usuarios totales por cohorte
cohort_totals AS (
    SELECT
        cohort_date,
        COUNT(DISTINCT user_id) AS users_cohort
    FROM cohort_events
    GROUP BY cohort_date
)

SELECT
    cc.cohort_date,
    cc.offset_week,
    ct.users_cohort,
    cc.users_cohort_offset,
    CAST(cc.users_cohort_offset * 1.0 / ct.users_cohort AS FLOAT) AS retention_rate
FROM cohort_counts cc
JOIN cohort_totals ct ON cc.cohort_date=ct.cohort_date
ORDER BY cohort_date, offset_week;

```


## 🤔💬 Momento de reflexionar en lo aprendido

----

Kahoot time

## 🚀 Para seguir aprendiendo :

---

- 📚 Vuelve a revisar este notebook y trata resolver por tu cuenta  nuevamente
- 💬 Recuerda que en Discord puedes dejar todos tus comentarios y dudas sobre el contenido del sprint en [`Discord`](https://discord.com/channels/1081207584104656986/1420849538196836472).
    - 📝 Si tienes preguntas sobre tu proyecto, usa el canal [`#project`](https://discord.com/channels/1081207584104656986/1420848813186351134) para recibir ayuda y compartir ideas.
    - 🤝 Aprovecha el espacio de `CoLearning` para aclarar tus dudas junto con otros estudiantes e instructores: [Co-Learning](https://discord.com/channels/1081207584104656986/1197953851391746119).
    - En tus preguntas recuerda etiquetar a `@Dataconsulta` y ubica tu pregunta de acuerdo a `Sprint/Capitulo/Seccion`
- 📅 ¿Necesitas ayuda personalizada? Puedes agendar una sesión `1:1` conmigo aquí: [1:1 Roman Castillo](https://scheduler.zoom.us/roman-castillo/1-1-roman-castillo).

- Por último hazme paro y responde la encuesta al final de la sesión, me sirve para poder ayudarte mejor 

¡Sigue practicando y no dudes en pedir apoyo cuando lo necesites! 💪✨